# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

<br>
**Note:** All record sets, fields, and columns referenced in this notebook use their unique `@id` values as specified in the dataset's Croissant schema.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Safely print dataset name and description
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all top-level record sets (by their `@id`) and show the fields (by their `@id`s) in each.

In [ ]:
# List available record set @ids and each record set's fields @ids
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - {field.get('@id', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will extract all tabular record sets (commonly those with data files) and load the main data table. If you're using this notebook, adapt the `RECORD_SET_IDS` list with the actual `@id`s of interest from above.

In [ ]:
# Based on the Croissant schema, let us automatically collect all record set IDs
RECORD_SET_IDS = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in RECORD_SET_IDS:
    try:
        print(f"\nLoading records from record set: {record_set_id}")
        # This will yield records as dictionaries with field @id as key
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records, columns: {list(df.columns)}")
        else:
            print("No records returned.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Let's pick the main record set for further analysis. Here we'll take the first one with data.
MAIN_RS_ID = None
for rsid, df in dataframes.items():
    if len(df) > 0:
        MAIN_RS_ID = rsid
        break
if MAIN_RS_ID is None:
    raise RuntimeError("No record sets with data found.")

print(f"\nColumns in record set {MAIN_RS_ID}:")
print(dataframes[MAIN_RS_ID].columns.tolist())
dataframes[MAIN_RS_ID].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Especially, we'll:
- Select a numeric field (column) by `@id`.
- Filter records where the field exceeds a threshold.
- Normalize the numeric field.
- Optionally, group by a categorical (non-numeric) field.

In [ ]:
# Let us heuristically select a numeric field from the main table.
df = dataframes[MAIN_RS_ID]

# Try to identify a likely numeric field (int or float).
numeric_field_id = None
for c in df.columns:
    # Try to infer if column is numeric
    try:
        series = pd.to_numeric(df[c], errors='coerce')
        if series.notna().sum() > 0 and series.dropna().astype(float).nunique() > 1:
            numeric_field_id = c
            break
    except Exception:
        continue

if numeric_field_id is None:
    print("No numeric field found.")
else:
    print(f"Using numeric field (by @id): {numeric_field_id}")

    # Choose a threshold: e.g. mean value
    threshold = pd.to_numeric(df[numeric_field_id], errors='coerce').mean()

    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_numeric = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_numeric - filtered_numeric.mean()) / filtered_numeric.std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a likely categorical variable
    group_field_id = None
    for c in reversed(df.columns):
        if c != numeric_field_id and df[c].nunique() < df.shape[0] // 2:
            group_field_id = c
            break

    if group_field_id is not None:
        print(f"\nGrouping by field (by @id): {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset. Here we plot the distribution of the selected numeric field, and if a group field was found, also group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), kde=True, bins=10)
    plt.title(f"Distribution of numeric field ({numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'filtered_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ci=None)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (filtered records)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² tabular dataset on second primary colorectal cancer survivors using the `mlcroissant` Python library, strictly referencing all Croissant entities by their `@id`. We demonstrated how to programmatically load and inspect record sets, extract tabular data, select fields for analysis by their `@id`, apply standard filtering and normalization, and visualize distributions. This workflow can be adapted to any FAIR/FAIR² dataset described by a Croissant schema.

**Key steps:**
- Identify record sets and fields (`@id`-referencing).
- Load and inspect data from a record set of interest.
- Conduct EDA using column `@id`s for reproducibility.
- Visualize results for interpretation.

You can adjust this template for other Croissant datasets by updating the dataset URL and field selections (always using `@id` references).